# OffsAid scam-language classifier: full training notebook
Fine-tunes `Davlan/afro-xlmr-small` as a **multi-label** classifier (7 labels) and exports a model plus per-label thresholds.

**Run order:** top to bottom on Colab (Runtime > Change runtime type > **T4 GPU**). Each cell has an *Expected output* note. If a cell errors, paste the error to Claude.

**Files you need to upload when asked:** `synthetic_train.csv`, `synthetic_val.csv` (from `generate_synthetic.py`, v5: 5,000 train and 1,000 validation rows). Later: `gold_test.csv` (real messages, never trained on).

**Honest reading of results:** synthetic validation scores will look very high (95%+). That only proves the pipeline works. The number that counts is the score on the **gold set of real messages** (Step 8).

**Why the last test showed F1 = 0.0:** with only 12 rows and a 0.5 threshold the model had not learned enough to push any score over 0.5, even though loss was falling (0.68 to 0.57). That is expected. It confirms training runs end to end. Ignore that number.

## Step 0. Install and check the GPU

In [1]:
!pip install -q "pyarrow<24" transformers datasets accelerate evaluate scikit-learn sentencepiece
import torch
print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

CUDA: True | Tesla T4


## Step 1. Load the data

In [2]:
from google.colab import files
import pandas as pd
print("Upload synthetic_train.csv and synthetic_val.csv")
up = files.upload()
LABELS = ["advance_fee_request","personal_account_payment","urgency_pressure",
          "secrecy","guaranteed_outcome","visa_travel_arranged","off_platform_move"]
train_df = pd.read_csv("synthetic_train.csv")
val_df   = pd.read_csv("synthetic_val.csv")
print(len(train_df), "train |", len(val_df), "val")
print(train_df[LABELS].sum().to_string())
print("rows with no label:", round((train_df[LABELS].sum(axis=1)==0).mean(), 2))

Upload synthetic_train.csv and synthetic_val.csv


Saving synthetic_val.csv to synthetic_val (5).csv
Saving synthetic_train.csv to synthetic_train (5).csv
4000 train | 800 val
advance_fee_request         1567
personal_account_payment     963
urgency_pressure             745
secrecy                      845
guaranteed_outcome           929
visa_travel_arranged         726
off_platform_move            779
rows with no label: 0.23


## Step 2. Load the model and tokenizer

In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
MODEL_ID = "Davlan/afro-xlmr-small"
id2label = {i:l for i,l in enumerate(LABELS)}
label2id = {l:i for i,l in id2label.items()}
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, num_labels=len(LABELS), problem_type="multi_label_classification",
    id2label=id2label, label2id=label2id)
print(round(sum(p.numel() for p in model.parameters())/1e6,1), "M parameters")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: Davlan/afro-xlmr-small
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


140.3 M parameters


## Step 3. Tokenise

In [4]:
from datasets import Dataset
def to_ds(df):
    ds = Dataset.from_pandas(df)
    def prep(b):
        enc = tokenizer(b["text"], truncation=True, max_length=128)
        enc["labels"] = [[float(b[l][i]) for l in LABELS] for i in range(len(b["text"]))]
        return enc
    return ds.map(prep, batched=True, remove_columns=list(df.columns))
train_ds, val_ds = to_ds(train_df), to_ds(val_df)
print(train_ds, val_ds, sep="\n")

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 4000
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 800
})


## Step 4. Metrics and trainer


In [5]:
import numpy as np
from sklearn.metrics import f1_score
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

def compute_metrics(p):
    probs = 1/(1+np.exp(-p.predictions))
    preds = (probs >= 0.5).astype(int)
    y = p.label_ids.astype(int)
    return {"f1_micro": f1_score(y, preds, average="micro", zero_division=0),
            "f1_macro": f1_score(y, preds, average="macro", zero_division=0)}

args = TrainingArguments(
    output_dir="out", num_train_epochs=5, per_device_train_batch_size=32,
    per_device_eval_batch_size=64, learning_rate=4e-5, weight_decay=0.01,
    warmup_steps=100, lr_scheduler_type="linear",
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model="f1_macro",
    fp16=True, logging_steps=25, report_to="none", seed=42)

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics)

## Step 5. Train

In [6]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro
1,0.249338,0.231309,0.850366,0.855655
2,0.056618,0.085734,0.959177,0.959745
3,0.031563,0.071992,0.955671,0.949738
4,0.022971,0.061531,0.963451,0.958594
5,0.021099,0.058445,0.962792,0.958432


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=625, training_loss=0.1289038353919983, metrics={'train_runtime': 312.3414, 'train_samples_per_second': 64.032, 'train_steps_per_second': 2.001, 'total_flos': 791259797208000.0, 'train_loss': 0.1289038353919983, 'epoch': 5.0})

## Step 6. Per-label report and threshold tuning

In [7]:
from sklearn.metrics import classification_report
pred = trainer.predict(val_ds)
val_probs = 1/(1+np.exp(-pred.predictions))
y_val = pred.label_ids.astype(int)

print(classification_report(y_val, (val_probs>=0.5).astype(int), target_names=LABELS, zero_division=0, digits=3))

# choose the threshold per label that maximises F1 on validation
thresholds = {}
for i,l in enumerate(LABELS):
    best_t, best_f = 0.5, -1
    for t in np.arange(0.2, 0.81, 0.05):
        f = f1_score(y_val[:,i], (val_probs[:,i]>=t).astype(int), zero_division=0)
        if f > best_f: best_t, best_f = float(round(t,2)), f
    thresholds[l] = best_t
print("tuned (ignore for now):", thresholds)
thresholds = {l: 0.5 for l in LABELS}   # safe default until tuned on the gold set
print("using:", thresholds)

                          precision    recall  f1-score   support

     advance_fee_request      0.997     1.000     0.999       338
personal_account_payment      0.995     1.000     0.998       210
        urgency_pressure      0.961     0.874     0.915       309
                 secrecy      1.000     0.991     0.996       227
      guaranteed_outcome      0.977     0.861     0.915       294
    visa_travel_arranged      0.812     1.000     0.896        95
       off_platform_move      1.000     1.000     1.000        54

               micro avg      0.972     0.946     0.959      1527
               macro avg      0.963     0.961     0.960      1527
            weighted avg      0.975     0.946     0.959      1527
             samples avg      0.831     0.811     0.815      1527

tuned (ignore for now): {'advance_fee_request': 0.7, 'personal_account_payment': 0.2, 'urgency_pressure': 0.3, 'secrecy': 0.4, 'guaranteed_outcome': 0.3, 'visa_travel_arranged': 0.75, 'off_platform_move': 

## Step 7. Save the model, thresholds and a zip

In [8]:
import json, shutil
SAVE_DIR = "offsaid_classifier_v4"
trainer.save_model(SAVE_DIR); tokenizer.save_pretrained(SAVE_DIR)
json.dump({"labels": LABELS, "thresholds": thresholds}, open(f"{SAVE_DIR}/thresholds.json","w"), indent=2)
shutil.make_archive(SAVE_DIR, "zip", SAVE_DIR)
from google.colab import drive
drive.mount("/content/drive")
shutil.copy(f"{SAVE_DIR}.zip", "/content/drive/MyDrive/")
print("saved to Drive: offsaid_classifier_v4.zip")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
saved to Drive: offsaid_classifier_v4.zip


## Step 8. Evaluate on the GOLD set (real messages). Do this once you have `gold_test.csv`

`gold_test.csv` needs the columns `text` plus the 7 label columns (0/1), same names as above. 100 to 200 real or realistic messages, **never** used for training or for tuning thresholds. Skip this cell until the file exists.

In [ ]:
import os
if os.path.exists("gold_test.csv") or input("Upload gold_test.csv now? (y/n) ").lower()=="y":
    if not os.path.exists("gold_test.csv"): files.upload()
    gold = pd.read_csv("gold_test.csv")
    g_ds = to_ds(gold)
    gp = trainer.predict(g_ds)
    g_probs = 1/(1+np.exp(-gp.predictions)); y_g = gp.label_ids.astype(int)
    th = np.array([thresholds[l] for l in LABELS])
    print(classification_report(y_g, (g_probs>=th).astype(int), target_names=LABELS, zero_division=0, digits=3))
    print("macro F1:", round(f1_score(y_g,(g_probs>=th).astype(int),average="macro",zero_division=0),3))
else:
    print("No gold set yet - skipping.")

*Expected:* a per-label table on real data. Expect clearly lower than synthetic (0.6 to 0.85 is realistic at first). The gap is the finding: it tells us which labels need more real or varied data.

## Step 9. Inference function (the interface the FastAPI service will use)

Three layers in one function: (1) the trained model, (2) **negation rules** that cancel `guaranteed_outcome` and `visa_travel_arranged` flags for sentences like "we can't guarantee the visa", (3) a **fee rule** that adds `advance_fee_request` when a payment word sits near a fee word (courier, customs, clearance, "medical money") and the sentence does not say nobody pays. Each flag now has `source`: `model` or `rule`. **This cell does not need retraining**: run it with the model you already have in memory.

In [10]:
import re

# ---------- negation rules (run AFTER the model; the model itself is unchanged) ----------
# words that mean "promise / guarantee" in English and Swahili
_GUAR = re.compile(r"\b(guarant\w*|promis\w*|assur\w*|\w*hakikish\w*|hatuhakikishi|uhakika|ahadi|ahidi\w*|\w*ahidi\w*)\b", re.I)
# a negator closely before it, e.g. "can't guarantee", "no one can promise", "hakuna mtu anaweza guarantee"
_NEG = re.compile(
    r"\b(can't|cant|cannot|can not|won't|wont|couldn't|unable|not|never|no one|nobody|none|"
    r"hakuna|hatuwezi|siwezi|hawezi|hawawezi|hapana)\b", re.I)
# a warning earlier in the same clause, e.g. "be careful of anyone who guarantees..."
_WARN = re.compile(r"\b(careful|beware|avoid|scams?|scammers?|fake|jihadhari|tahadhari|usiamini|wadanganyifu)\b", re.I)
# clause boundaries: punctuation or "but / ila / lakini"
_BREAK = re.compile(r"[,;:.!?\n]|\b(?:but|however|ila|lakini)\b", re.I)
# visa / travel words
_VISA = re.compile(r"\b(visa|visas|ticket|tickets|tiketi|flight|permit|kibali|hotel|hoteli|travel|safari|passport)\b", re.I)

def negated_guarantee(text, max_gap=3):
    """True only if the text uses a guarantee/promise word and EVERY such use is negated or warned about.
    No guarantee word, or at least one un-negated one -> False (the model decides)."""
    hits = list(_GUAR.finditer(text))
    if not hits:
        return False
    for h in hits:
        start = max((m.end() for m in _BREAK.finditer(text[:h.start()])), default=0)
        before = text[start:h.start()]
        negated = any(len(before[n.end():].split()) <= max_gap for n in _NEG.finditer(before))
        if not negated and _WARN.search(before):
            negated = True
        if not negated:
            return False
    return True

def negated_visa_claim(text):
    """True if every clause that mentions visa/ticket/flight/permit... only does so inside a negated
    guarantee ("we can't guarantee the visa"). A clause like "your visa is approved" -> False."""
    clauses = [c for c in _BREAK.split(text) if c and c.strip()]
    hits = [c for c in clauses if _VISA.search(c)]
    return bool(hits) and all(negated_guarantee(c) for c in hits)

# label -> rule that cancels a flag
SUPPRESS = {"guaranteed_outcome": negated_guarantee, "visa_travel_arranged": negated_visa_claim}

# ---------- fee rule: a payment word near a fee word, unless the sentence says nobody pays ----------
_PAYV = {"pay","paid","paying","payment","payments","send","sending","settle","settled","cover","covering","clear","cleared",
         "transfer","pays","lipa","lipia","lipwa","tuma","nitumie","weka","kulipa","malipo","ulipe","ilipwe","ilipiwe","kulipwa"}
# strong fee words count with any payment word; weak ones (courier, delivery...) only with a real pay verb,
# so "send the contract by courier" or "send your delivery address" is not a fee request
_FEEN = {"fee","fees","charge","charges","cost","costs","customs","clearance","duty","ada","gharama","ushuru","forodha"}
_WEAKN = {"courier","dhl","shipping","delivery","insurance","tax"}
_TRUEPAY = {"pay","paid","paying","payment","payments","pays","settle","settled","cover","covering","clear","cleared",
            "lipa","lipia","lipwa","kulipa","ulipe","ilipwe","ilipiwe","kulipwa","malipo"}
# doctor / medical only count with a real pay verb ("pay the doctor"), never with "send" ("send your medical results")
_PAYONLY_V = {"pay","paid","paying","payment","payments","lipa","lipwa","kulipa","ulipe","ilipwe","kulipwa"}
_PAYONLY_N = {"doctor","medical","medicals","daktari","vipimo"}
_FEE_PHRASE = re.compile(r"\b(medical|vipimo|embassy|visa)\s+(money|fee|fees|payment)\b|\bpesa\s+ya\s+(medical|vipimo|courier|customs|clearance|docs|documents|embassy|dhl)\b|\bmedical money\b", re.I)
_NOT_A_REQUEST = re.compile(
    r"\b(no|zero|without)\s+(\w+\s+){0,2}(fee|fees|charge|charges|cost|costs|payment|payments)\b|\bno cost\b|\bfree\b|\bfree of charge\b|"
    r"\b(do|does|did) not (ask|need|require|charge)\b|\b(don't|doesn't|never) (ask|charge)\b|\bnever (ask|send|pay)\b|"
    r"\b(you|players?) (do not|don't|will not|won't|never) pay\b|\bpay(s)? nothing\b|\bpaid by\b|\bnot (pay|paying)\b|"
    r"\b(club|club's|company|buying club)\s+(will\s+)?(pay|pays|paying|cover|covers|covering|handle|handles|reimburse\w*)\b|"
    r"\b(do not|don't|never)\s+(send|pay|transfer|share)\b|\bhakuna malipo\b|\bbila\s+(malipo|gharama)\b|\bhulipi\b|\bhaulipi\b|"
    r"\b(usitume|usilipe|usitume)\b|\binalipa\b|\bitalipa\b|\bitakulipia\b|\binalipwa\b|\bzinalipwa\b|\bhaitakuomba\b|\bhaiulizi\b|\bbure\b", re.I)

def fee_rule(text, max_gap=7):
    """True if a clause has a payment word within max_gap words of a fee word and the sentence does not say
    nobody pays. Deliberately simple: it only ADDS advance_fee_request when the model scored it below threshold."""
    text = re.sub(r"(?<=\d),(?=\d)", "", text)   # 20,000 -> 20000 so the comma is not a clause break
    if _NOT_A_REQUEST.search(text):
        return False
    if _FEE_PHRASE.search(text):
        return True
    for clause in _BREAK.split(text):
        words = re.findall(r"[a-zA-Z']+", clause.lower())
        v = [i for i, w in enumerate(words) if w in _PAYV]
        n = [i for i, w in enumerate(words) if w in _FEEN]
        if any(abs(i - j) <= max_gap for i in v for j in n):
            return True
        tv = [i for i, w in enumerate(words) if w in _TRUEPAY]
        wn = [i for i, w in enumerate(words) if w in _WEAKN]
        if any(abs(i - j) <= max_gap for i in tv for j in wn):
            return True
        pv = [i for i, w in enumerate(words) if w in _PAYONLY_V]
        pn = [i for i, w in enumerate(words) if w in _PAYONLY_N]
        if any(abs(i - j) <= max_gap for i in pv for j in pn):
            return True
    return False

import torch
model.eval()
def classify(messages):
    """messages: [{"sender":"agent","text":"..."}] -> {"flags":[{"label","quote","score","source"}]}
    source = "model" (the classifier) or "rule" (fee rule added because the model scored below threshold)."""
    best, via_rule = {}, {}   # label -> (rank, quote)
    for m in messages:
        sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+|\n+", m["text"]) if s.strip()]
        units = sents + ([m["text"]] if len(sents) > 1 else [])   # sentences and the whole message
        enc = tokenizer(units, return_tensors="pt", truncation=True, max_length=128, padding=True).to(model.device)
        with torch.no_grad():
            probs = torch.sigmoid(model(**enc).logits).cpu().numpy()
        for u, p in zip(units, probs):
            is_sentence = (u != m["text"]) or len(sents) == 1
            for i, l in enumerate(LABELS):
                if p[i] < thresholds[l]:
                    continue
                # negation rules: "we can't guarantee the visa", "be careful of anyone who guarantees..." are not claims
                if l in SUPPRESS and SUPPRESS[l](u):
                    continue
                rank = (1 if is_sentence else 0, float(p[i]))
                if l not in best or rank > best[l][0]:
                    best[l] = (rank, u)
            # fee rule: only adds a flag the model did not raise for this sentence, and only on sentence units
            if is_sentence and fee_rule(u) and "advance_fee_request" not in via_rule:
                via_rule["advance_fee_request"] = u
    flags = [{"label": l, "quote": q, "score": round(r[1], 3), "source": "model"} for l, (r, q) in best.items()]
    if "advance_fee_request" in via_rule and "advance_fee_request" not in best:
        flags.append({"label": "advance_fee_request", "quote": via_rule["advance_fee_request"], "score": 0.5, "source": "rule"})
    return {"flags": flags}

# Fresh messages, worded differently from anything in the training clauses. "want" = what a correct run shows.
tests = [
  ("Congratulations bro! The club has picked you. Only thing left is paying the courier for your papers.", "guaranteed_outcome, advance_fee_request"),
  ("Hi, DHL is holding your contract until the delivery fee is paid. Send it today.", "advance_fee_request, urgency_pressure"),
  ("Hakuna mtu atakuhakikishia visa, ni ubalozi unaoamua. Trial ni bure.", "nothing"),
  ("We cannot promise any visa, the embassy decides. Come to the open trial, no charge.", "nothing"),
  ("The doctor must be paid before your medical, send the money to my sister's number 0722 111 222.", "advance_fee_request, personal_account_payment"),
  ("Pole kwa kuchelewa. Tuma ile pesa ya customs ndio documents zitoke, usiwaambie wengine.", "advance_fee_request, secrecy"),
  ("Congratulations on your trial. The coach will go through all players and we will email results next week.", "nothing"),
  ("Your flight is booked and your visa is approved, just settle the clearance charges.", "visa_travel_arranged, advance_fee_request"),
  ("We can't guarantee the visa, but your contract is 100% guaranteed.", "guaranteed_outcome only (NOT visa)"),
  ("Your trial is on Saturday at 9am. Bring boots and ID. There is no fee.", "nothing"),
  ("The club will pay for your courier and documents, you pay nothing.", "nothing"),
  ("Please send your medical results and passport copy by email.", "nothing"),
]
for text, want in tests:
    got = [f["label"] + (" (rule)" if f["source"] == "rule" else "") for f in classify([{"sender": "agent", "text": text}])["flags"]]
    print(text); print("  want:", want); print("  got :", got or "nothing"); print()

Congratulations bro! The club has picked you. Only thing left is paying the courier for your papers.
  want: guaranteed_outcome, advance_fee_request
  got : ['guaranteed_outcome', 'advance_fee_request (rule)']

Hi, DHL is holding your contract until the delivery fee is paid. Send it today.
  want: advance_fee_request, urgency_pressure
  got : ['urgency_pressure', 'visa_travel_arranged', 'advance_fee_request (rule)']

Hakuna mtu atakuhakikishia visa, ni ubalozi unaoamua. Trial ni bure.
  want: nothing
  got : nothing

We cannot promise any visa, the embassy decides. Come to the open trial, no charge.
  want: nothing
  got : nothing

The doctor must be paid before your medical, send the money to my sister's number 0722 111 222.
  want: advance_fee_request, personal_account_payment
  got : ['personal_account_payment', 'advance_fee_request (rule)']

Pole kwa kuchelewa. Tuma ile pesa ya customs ndio documents zitoke, usiwaambie wengine.
  want: advance_fee_request, secrecy
  got : ['secrecy

## Step 9b. Raw scores for the messages the model has been missing

In [11]:
# Step 9b. Show the raw score for every label, so a miss is visible as "0.31 (close)" or "0.01 (far)"
def show_scores(text):
    enc = tokenizer([text], return_tensors="pt", truncation=True, max_length=128).to(model.device)
    with torch.no_grad():
        p = torch.sigmoid(model(**enc).logits)[0].cpu().numpy()
    print(text)
    print("  " + " | ".join(f"{l.split('_')[0]} {p[i]:.2f}" for i, l in enumerate(LABELS)))
    print()

for t in [
  "Congratulations! Your contract is waiting. Just cover the courier charges for the documents.",
  "Brother the club needs the medical money before Friday, send to my wife's number.",
  "Your contract is guaranteed, just send the fee today.",
  "Just cover the courier charges for the documents.",
  "Your contract is waiting.",
]:
    show_scores(t)

Congratulations! Your contract is waiting. Just cover the courier charges for the documents.
  advance 0.04 | personal 0.01 | urgency 0.01 | secrecy 0.01 | guaranteed 0.05 | visa 0.03 | off 0.01

Brother the club needs the medical money before Friday, send to my wife's number.
  advance 0.10 | personal 0.81 | urgency 0.16 | secrecy 0.01 | guaranteed 0.01 | visa 0.02 | off 0.01

Your contract is guaranteed, just send the fee today.
  advance 0.86 | personal 0.03 | urgency 0.84 | secrecy 0.02 | guaranteed 0.94 | visa 0.03 | off 0.01

Just cover the courier charges for the documents.
  advance 0.04 | personal 0.01 | urgency 0.01 | secrecy 0.01 | guaranteed 0.01 | visa 0.02 | off 0.01

Your contract is waiting.
  advance 0.02 | personal 0.01 | urgency 0.30 | secrecy 0.01 | guaranteed 0.79 | visa 0.05 | off 0.01



## Step 10. Check it runs on CPU (this is how we will deploy)

In [12]:
import time
cpu = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR).to("cpu").eval()
enc = tokenizer(["Send KES 25,000 today to secure your slot. Don't tell anyone."], return_tensors="pt")
t0 = time.time()
with torch.no_grad(): cpu(**enc)
print("CPU latency (first call):", round(time.time()-t0, 2), "s")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

CPU latency (first call): 0.34 s


In [13]:
import time
msgs = [{"sender": "agent", "text": t} for t in [
    "Hi bro, tuma KES 15,000 kwa mpesa yangu 0712 345 678 ndio slot ikae. Usimwambie mtu.",
    "Your trial is on Saturday at 9am. Bring boots and ID. There is no fee.",
    "Visa yako imepita. Tuhamie Telegram, WhatsApp si safe.",
    "Congratulations! Your contract is waiting. Just cover the courier charges.",
    "We can't guarantee anything, but come for the trial on Saturday, it is free."]]
model.to("cpu"); model.eval()          # classify() uses `model`, so move it to CPU for a fair test
classify(msgs[:1])                     # warm-up, not timed
t0 = time.time()
for _ in range(20): classify(msgs[:1])
print("one message, average:", round((time.time() - t0) / 20, 2), "s")
t0 = time.time(); classify(msgs)
print("5-message chat:", round(time.time() - t0, 2), "s")

one message, average: 0.34 s
5-message chat: 1.06 s


## What comes next
1. Build the **gold set** (100 to 200 real messages) and run Step 8.
2. Review the Swahili/Sheng clauses in `generate_synthetic.py` with a native speaker and add real phrasings.
3. Look at the gold-set errors, add targeted examples, retrain.
4. Add the rules layer and the LLM few-shot layer, then combine the three signals.
5. Wrap `classify()` in the FastAPI service.